# Agente Educacional com LangChain + LangGraph
> **Do Piauí para o Mundo** — Palestra "LangChain na Sala de Aula"

Este notebook acompanha a palestra e mostra **cada etapa do agente funcionando**.
Execute célula por célula e observe o que acontece em cada passo.

---

## O que vamos construir?

```
Pergunta do aluno
      │
      ▼
  [RAG] Busca contexto na base de dados
      │
      ▼
  [Prompt] Monta a instrução para o modelo
      │
      ▼
  [LLM] Soberano 1.1 gera a resposta
      │
      ▼
  [Guardrail] Valida se a resposta é segura
      │
      ▼
 Resposta confiável
```

---

## 0. Configuração

Antes de começar, instale as dependências e configure as variáveis de ambiente.

```bash
pip install -e .
cp .env.example .env   # edite com sua chave de API
```

In [ ]:
import sys
sys.path.insert(0, "../src")  # aponta para o código-fonte do projeto

# Verifica se consegue importar o pacote
import agente_edu
print("✓ Pacote importado com sucesso!")

---
## 1. RAG — Recuperação de Informação

**RAG** = *Retrieval-Augmented Generation* (Geração Aumentada por Recuperação)

A ideia central: **em vez de o modelo "lembrar" os dados, ele consulta uma fonte**.
Isso elimina alucinações sobre os dados do domínio.

### 1.1 Carregar os dados

In [ ]:
from agente_edu.rag.ingest import carregar

documentos = carregar("../data/pib_piaui.csv")

print(f"Total de documentos carregados: {len(documentos)}")
print()
print("Exemplo de documento:")
print(f"  page_content: {documentos[0].page_content}")
print(f"  metadata:     {documentos[0].metadata}")

Cada linha do CSV vira um `Document` com:
- `page_content`: o texto que o LLM vai receber como contexto
- `metadata`: os dados originais (para filtragem ou rastreabilidade)

### 1.2 Criar o índice vetorial (embeddings)

In [ ]:
from agente_edu.rag.ingest import indexar

# Isso pode demorar alguns segundos na primeira vez
# (baixa o modelo de embeddings se ainda não tiver)
vectorstore = indexar(documentos)

print("✓ Índice vetorial criado!")
print(f"  Tipo: {type(vectorstore).__name__}")

### 1.3 Busca semântica

Agora podemos buscar documentos por **significado**, não por palavra-chave exata.

In [ ]:
pergunta = "Qual estado tem o maior PIB do Nordeste?"

resultados = vectorstore.similarity_search(pergunta, k=3)

print(f"Pergunta: {pergunta}")
print(f"\nTop {len(resultados)} documentos mais relevantes:")
for i, doc in enumerate(resultados, 1):
    print(f"\n  [{i}] {doc.page_content}")

In [ ]:
# Testando com uma pergunta que não tem resposta nos dados
pergunta_sem_resposta = "Qual é a capital da França?"

resultados = vectorstore.similarity_search(pergunta_sem_resposta, k=3)

print(f"Pergunta: {pergunta_sem_resposta}")
print(f"\nDocumentos retornados (mesmo sem resposta exata):")
for i, doc in enumerate(resultados, 1):
    print(f"  [{i}] {doc.page_content}")

print("\n⚠ O RAG sempre retorna documentos — por isso o guardrail é importante!")

---
## 2. Prompt — Instruindo o Modelo

Um `PromptTemplate` é um template com variáveis que preenchemos na hora de usar.
No LangChain, prompts são objetos reutilizáveis e compináveis.

In [ ]:
from agente_edu.chains.prompt import PERSONAS, montar_prompt

# Ver as personas disponíveis
print("Personas disponíveis:")
for nome, descricao in PERSONAS.items():
    print(f"  {nome}: {descricao[:60]}...")

In [ ]:
from agente_edu.guardrails.rules import REGRA

prompt = montar_prompt("ensino_medio")

contexto_exemplo = "estado: Piauí, pib_2023_bilhoes: 63.4, idh_2022: 0.715"
pergunta_exemplo = "Qual é o PIB do Piauí?"

# Formata o prompt com os valores reais
prompt_formatado = prompt.format(
    contexto=contexto_exemplo,
    pergunta=pergunta_exemplo,
    regra=REGRA,
)

print("=" * 60)
print("PROMPT QUE VAI PARA O MODELO:")
print("=" * 60)
print(prompt_formatado)

In [ ]:
# Comparando o mesmo prompt com personas diferentes
for persona in ["crianca", "vestibular"]:
    p = montar_prompt(persona)
    texto = p.format(contexto=contexto_exemplo, pergunta=pergunta_exemplo, regra=REGRA)
    print(f"--- Persona: {persona} ---")
    print(texto[:300], "...")
    print()

---
## 3. LLM — O Modelo de Linguagem

O **Soberano 1.1** é um LLM em português. Aqui usamos uma classe customizada
que extende `BaseChatModel` do LangChain — isso o torna compatível com
qualquer chain ou ferramenta do ecossistema.

In [ ]:
from agente_edu.llm.soberano import Soberano
from agente_edu.config import settings

print("Configurações do modelo:")
print(f"  URL da API:  {settings.soberano_api_base_url}")
print(f"  Modelo:      {settings.soberano_model}")
print(f"  API key:     {'configurada ✓' if settings.soberano_api_key else 'NÃO configurada ✗'}")

modelo = Soberano()
print(f"\nTipo LangChain: {modelo._llm_type}")
print("✓ Instância criada. Chamadas só acontecem ao invocar a chain.")

---
## 4. Chain — Ligando as Peças

No LangChain, o operador `|` conecta componentes em sequência.
Cada componente recebe a saída do anterior.

```
prompt | modelo | parser
  ↑         ↑        ↑
formata   chama    extrai
o texto   a API    o texto
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from agente_edu.chains.prompt import montar_prompt
from agente_edu.llm.soberano import Soberano
from agente_edu.guardrails.rules import REGRA

# Construindo a chain manualmente para entender cada peça
prompt   = montar_prompt("ensino_medio")
modelo   = Soberano()
parser   = StrOutputParser()

chain = prompt | modelo | parser

print("Chain criada:")
print(f"  {type(prompt).__name__} | {type(modelo).__name__} | {type(parser).__name__}")

In [ ]:
# Executando a chain (requer API key configurada)
from agente_edu.rag.retriever import Retriever

retriever = Retriever()
pergunta = "Qual é o PIB do Piauí?"
contexto = retriever.buscar(pergunta, k=3)

print("Contexto recuperado:")
print(contexto)
print()

try:
    resposta = chain.invoke({"contexto": contexto, "pergunta": pergunta, "regra": REGRA})
    print("Resposta do modelo:")
    print(resposta)
except Exception as e:
    print(f"⚠ Erro ao chamar a API: {e}")
    print("Configure SOBERANO_API_KEY no arquivo .env para testar com o modelo real.")

---
## 5. Guardrails — Segurança em Dois Níveis

Guardrails garantem que o agente **não invente respostas**.
Há dois níveis que trabalham juntos:

In [ ]:
from agente_edu.guardrails.rules import REGRA, FRASE_FALLBACK, validar

print("NÍVEL 1 — Guardrail no Prompt (instrui o modelo):")
print(REGRA)
print()
print("NÍVEL 2 — Guardrail no Código (valida programaticamente):")
print("  def validar(resposta, contexto) -> str: ...")

In [ ]:
# Simulando o guardrail nível 2

# Caso 1: contexto presente → resposta passa
resposta = "O PIB do Piauí é 63.4 bilhões."
contexto = "estado: Piauí, pib_2023_bilhoes: 63.4"
resultado = validar(resposta, contexto)
print(f"Caso 1 (com contexto):")
print(f"  Resposta do modelo: {resposta}")
print(f"  Resultado:          {resultado}")
print()

# Caso 2: contexto vazio → guardrail bloqueia, mesmo que o modelo tenha respondido
resposta_inventada = "O PIB do Piauí cresceu 20% em 2024."
resultado = validar(resposta_inventada, contexto="")
print(f"Caso 2 (sem contexto):")
print(f"  Resposta do modelo: {resposta_inventada}")
print(f"  Resultado:          {resultado}")
print()
print(f"⚠ O guardrail substituiu a resposta, independente do que o modelo disse.")

---
## 6. LangGraph — Orquestrando com um Grafo

O LangGraph permite criar **fluxos com estado e decisões condicionais**.
Em vez de uma chain linear, temos um grafo com nós e arestas.

O grafo deste agente:
```
recuperar → decidir() → responder → FIM
                     ↘
                      buscar_mais → responder → FIM
```

In [ ]:
from agente_edu.graph.agent_graph import Estado, decidir

# Testando o roteador isoladamente
print("Testando a função de decisão (decidir):")
print()

# Caso 1: há contexto → vai para responder
estado_com_contexto: Estado = {
    "pergunta": "Qual o PIB do Piauí?",
    "contexto": "Piauí, pib_2023_bilhoes: 63.4",
    "resposta": "",
    "tentativas": 0,
}
print(f"Estado com contexto → próximo nó: '{decidir(estado_com_contexto)}'")

# Caso 2: sem contexto, primeira tentativa → vai para buscar_mais
estado_sem_contexto: Estado = {
    "pergunta": "Qual o PIB do Piauí?",
    "contexto": "",
    "resposta": "",
    "tentativas": 0,
}
print(f"Estado sem contexto (1ª vez)  → próximo nó: '{decidir(estado_sem_contexto)}'")

# Caso 3: sem contexto, segunda tentativa → desiste e vai para responder
estado_segunda_tentativa = {**estado_sem_contexto, "tentativas": 1}
print(f"Estado sem contexto (2ª vez)  → próximo nó: '{decidir(estado_segunda_tentativa)}'")

In [ ]:
from agente_edu.graph.agent_graph import construir_grafo

grafo = construir_grafo()

print("Grafo compilado com sucesso!")
print(f"Nós: {list(grafo.nodes)}")

In [ ]:
# Executando o grafo completo (requer API key)
try:
    resultado = grafo.invoke({
        "pergunta": "Qual é o PIB do Piauí?",
        "contexto": "",
        "resposta": "",
        "tentativas": 0,
    })
    print("Resultado do grafo:")
    print(f"  Pergunta:   {resultado['pergunta']}")
    print(f"  Contexto:   {resultado['contexto'][:100]}..." if resultado['contexto'] else "  Contexto:   (vazio)")
    print(f"  Tentativas: {resultado['tentativas']}")
    print(f"  Resposta:   {resultado['resposta']}")
except Exception as e:
    print(f"⚠ Requer API key: {e}")

---
## 7. Pipeline Completo — Debug Mode

A função `responder_com_detalhes` expõe cada etapa da pipeline.
É especialmente útil para entender o que aconteceu em cada chamada.

In [ ]:
from agente_edu.chains.qa_chain import responder_com_detalhes
from agente_edu.rag.retriever import Retriever

retriever = Retriever()
pergunta = "Qual é o IDH do Maranhão?"
contexto = retriever.buscar(pergunta, k=3)

print("Contexto recuperado pelo RAG:")
print(contexto)
print()

try:
    info = responder_com_detalhes(pergunta, contexto, persona="ensino_medio")

    print("=" * 60)
    print("DETALHES DO PIPELINE")
    print("=" * 60)
    print(f"Persona:             {info.persona}")
    print(f"Guardrail aplicado:  {info.guardrail_aplicado}")
    print()
    print("Prompt enviado ao modelo:")
    print(info.prompt_montado)
    print()
    print("Resposta bruta do modelo:")
    print(info.resposta_bruta)
    print()
    print("Resposta final (após guardrail):")
    print(info.resposta_final)
    print()
    print("Passos executados:")
    for passo in info.passos:
        print(f"  • {passo}")
except Exception as e:
    print(f"⚠ Requer API key configurada: {e}")

---
## 8. Exercícios para o aluno

Agora é sua vez! Tente:

1. **Adicionar uma nova persona**: edite `chains/prompt.py` e adicione uma persona `universitario`.
   Teste-a neste notebook.

2. **Adicionar dados novos**: coloque um novo CSV em `data/` e altere `DATA_PATH` no `.env`.
   Reconstrua o índice e faça uma pergunta sobre os novos dados.

3. **Explorar o roteador**: modifique `decidir` em `graph/agent_graph.py` para
   usar `tentativas < 2` (três tentativas). O que muda?

4. **Testar os guardrails**: tente enganar o agente fazendo uma pergunta cujo
   assunto não está no CSV. O guardrail funciona?

---

## Resumo dos conceitos

| Conceito | Arquivo | O que faz |
|---|---|---|
| Document | `rag/ingest.py` | Unidade de texto com metadados |
| Embeddings | `rag/ingest.py` | Converte texto em vetor numérico |
| VectorStore | `rag/ingest.py` | Banco de dados de vetores |
| Retriever | `rag/retriever.py` | Busca semântica no índice |
| PromptTemplate | `chains/prompt.py` | Template parametrizável de instrução |
| BaseChatModel | `llm/soberano.py` | Interface padrão de LLM no LangChain |
| Chain (`\|`) | `chains/qa_chain.py` | Sequência de componentes |
| Guardrail | `guardrails/rules.py` | Validação de segurança |
| TypedDict | `graph/agent_graph.py` | Estado do grafo |
| StateGraph | `graph/agent_graph.py` | Grafo com estado (LangGraph) |
| add_conditional_edges | `graph/agent_graph.py` | Roteamento dinâmico no grafo |